# Feature engineering: spatial and temporal features

This notebook builds the engineered features for demand forecasting, including spatial features (distance to downtown, zone clustering), cyclical temporal encoding, and derived indicators.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('../data/ride_demand.csv')
print(f'Dataset: {df.shape}')
df.head()

## Spatial feature: distance to downtown

We compute the haversine distance from each record's coordinates to the Calgary downtown core (51.0477, -114.0630). Downtown proximity is a strong predictor of ride demand.

In [ ]:
DOWNTOWN_LAT, DOWNTOWN_LON = 51.0477, -114.0630

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_r, lat2_r = np.radians(lat1), np.radians(lat2)
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df['distance_to_downtown'] = df.apply(
    lambda r: haversine_km(r['latitude'], r['longitude'], DOWNTOWN_LAT, DOWNTOWN_LON), axis=1
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df['distance_to_downtown'], df['demand_count'], alpha=0.1, s=5, c='#3B6FD4')
ax.set_xlabel('Distance to downtown (km)')
ax.set_ylabel('Demand count')
ax.set_title('Distance to downtown vs demand')
plt.tight_layout()
plt.show()

print(f'Distance range: {df["distance_to_downtown"].min():.2f} - {df["distance_to_downtown"].max():.2f} km')
print(f'Correlation with demand: {df["distance_to_downtown"].corr(df["demand_count"]):.3f}')

## Spatial feature: zone clustering with KMeans

We cluster zones by geographic coordinates to create spatial groupings. This captures regional patterns that are not captured by individual lat/lon values.

In [ ]:
coords = df[['latitude', 'longitude']].values

# Find optimal k using inertia (elbow method)
inertias = []
k_range = range(2, 12)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(coords)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_range, inertias, 'bo-', color='#3B6FD4')
ax.axvline(x=6, color='#E8C230', linestyle='--', label='k=6 (selected)')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow method for zone clustering')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df['zone_cluster'] = kmeans.fit_predict(coords)

fig, ax = plt.subplots(figsize=(10, 10))
cluster_colors = ['#3B6FD4', '#E8C230', '#22c55e', '#ef4444', '#8b5cf6', '#f97316']
for c in range(6):
    mask = df['zone_cluster'] == c
    ax.scatter(df.loc[mask, 'longitude'], df.loc[mask, 'latitude'],
              s=5, alpha=0.3, c=cluster_colors[c], label=f'Cluster {c}')

# Plot centroids
for i, (lat, lon) in enumerate(kmeans.cluster_centers_):
    ax.scatter(lon, lat, s=200, c=cluster_colors[i], edgecolors='black', linewidths=2, marker='*', zorder=5)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Zone clusters (k=6) based on geographic coordinates')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

print('Cluster sizes:')
print(df['zone_cluster'].value_counts().sort_index())
print(f'\nMean demand by cluster:')
print(df.groupby('zone_cluster')['demand_count'].mean().round(2))

## Temporal feature: cyclical hour encoding

Hours of the day are cyclical -- hour 23 is close to hour 0. Encoding them as sine/cosine preserves this circular relationship.

In [ ]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Raw hour encoding
axes[0].scatter(df['hour'], df['demand_count'], alpha=0.05, s=5, c='#3B6FD4')
axes[0].set_xlabel('Hour (raw)')
axes[0].set_ylabel('Demand')
axes[0].set_title('Raw hour vs demand')

# Cyclical encoding visualization
hours = np.arange(24)
axes[1].plot(np.sin(2 * np.pi * hours / 24), np.cos(2 * np.pi * hours / 24), 'o-', color='#3B6FD4')
for h in hours:
    axes[1].annotate(str(h), (np.sin(2 * np.pi * h / 24), np.cos(2 * np.pi * h / 24)),
                    fontsize=8, ha='center', va='bottom')
axes[1].set_xlabel('sin(hour)')
axes[1].set_ylabel('cos(hour)')
axes[1].set_title('Cyclical hour encoding')
axes[1].set_aspect('equal')

# Sine/cosine vs demand
axes[2].scatter(df['hour_sin'], df['demand_count'], alpha=0.05, s=5, c='#3B6FD4', label='sin')
axes[2].scatter(df['hour_cos'], df['demand_count'], alpha=0.05, s=5, c='#E8C230', label='cos')
axes[2].set_xlabel('Encoded value')
axes[2].set_ylabel('Demand')
axes[2].set_title('Cyclical encoding vs demand')
axes[2].legend()

plt.tight_layout()
plt.show()

## Temporal feature: cyclical day and month encoding

In [ ]:
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Day of week cyclical
days = np.arange(7)
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[0].plot(np.sin(2 * np.pi * days / 7), np.cos(2 * np.pi * days / 7), 'o-', color='#3B6FD4')
for d in days:
    axes[0].annotate(day_names[d], (np.sin(2 * np.pi * d / 7), np.cos(2 * np.pi * d / 7)),
                    fontsize=9, ha='center', va='bottom')
axes[0].set_xlabel('sin(day)')
axes[0].set_ylabel('cos(day)')
axes[0].set_title('Cyclical day-of-week encoding')
axes[0].set_aspect('equal')

# Month cyclical
months = np.arange(1, 13)
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1].plot(np.sin(2 * np.pi * months / 12), np.cos(2 * np.pi * months / 12), 'o-', color='#E8C230')
for m in months:
    axes[1].annotate(month_names[m - 1], (np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)),
                    fontsize=9, ha='center', va='bottom')
axes[1].set_xlabel('sin(month)')
axes[1].set_ylabel('cos(month)')
axes[1].set_title('Cyclical month encoding')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

## Derived indicators: rush hour and weekend flags

In [ ]:
df['is_rush_hour'] = ((df['hour'] >= 7) & (df['hour'] <= 9) |
                      (df['hour'] >= 16) & (df['hour'] <= 18)).astype(int)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

rush_demand = df.groupby('is_rush_hour')['demand_count'].mean()
axes[0].bar(['Off-peak', 'Rush hour'], rush_demand.values, color=['#3B6FD4', '#E8C230'], edgecolor='black')
axes[0].set_title('Demand: rush hour vs off-peak')
axes[0].set_ylabel('Average demand')
for i, v in enumerate(rush_demand.values):
    axes[0].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

weekend_demand = df.groupby('is_weekend')['demand_count'].mean()
axes[1].bar(['Weekday', 'Weekend'], weekend_demand.values, color=['#3B6FD4', '#E8C230'], edgecolor='black')
axes[1].set_title('Demand: weekday vs weekend')
axes[1].set_ylabel('Average demand')
for i, v in enumerate(weekend_demand.values):
    axes[1].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Feature matrix overview

In [ ]:
feature_cols = [
    'latitude', 'longitude', 'hour_sin', 'hour_cos',
    'day_sin', 'day_cos', 'month_sin', 'month_cos',
    'is_holiday', 'temperature', 'precipitation',
    'event_nearby', 'population_density', 'num_restaurants',
    'transit_stops_nearby', 'distance_to_downtown', 'zone_cluster',
    'is_rush_hour', 'is_weekend',
]

print(f'Total features: {len(feature_cols)}')
print(f'\nFeature list:')
for i, f in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {f}')

print(f'\nFeature matrix shape: ({len(df)}, {len(feature_cols)})')
print(f'Target: demand_count')
print(f'\nFeature statistics:')
df[feature_cols].describe().round(2)

## Updated correlation with engineered features

In [ ]:
all_cols = feature_cols + ['demand_count']
corr = df[all_cols].corr()

target_corr = corr['demand_count'].drop('demand_count').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E8C230' if v > 0 else '#3B6FD4' for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='black')
ax.set_xlabel('Correlation with demand_count')
ax.set_title('Feature correlations with target')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Top positive correlations:')
print(target_corr.head(5).round(3))
print('\nTop negative correlations:')
print(target_corr.tail(5).round(3))

## Summary

Engineered features added to the dataset:

1. **distance_to_downtown** -- haversine distance from each record to downtown Calgary; shows negative correlation with demand, confirming downtown zones attract more rides
2. **zone_cluster** -- KMeans (k=6) clusters on lat/lon capturing regional groupings; different clusters exhibit distinct demand levels
3. **hour_sin / hour_cos** -- cyclical encoding of hour preserving circular proximity between hour 23 and hour 0
4. **day_sin / day_cos** -- cyclical encoding of day of week
5. **month_sin / month_cos** -- cyclical encoding of month capturing seasonal patterns
6. **is_rush_hour** -- binary flag for morning (7-9) and evening (16-18) rush hours; rush hour demand is significantly higher
7. **is_weekend** -- binary flag for Saturday and Sunday

The final feature matrix has 19 features across spatial, temporal, weather, infrastructure, and event dimensions.